# Data Generation

This notebook simulates pp collisions with $\sqrt{s} = 510 \: \mathrm{GeV}$ at the STAR experiment and stores them into a ROOT tree. Afterwards, detector acceptance and smearing is simulated a new dataset is obtained.

### ROOT Tree Generation Using PYTHIA8

At first, it is necessary to import os, ROOT, and load PYTHIA8.

In [ ]:
import os
import ROOT

ROOT.gSystem.Load("libpythia8") # load PYTHIA8 into ROOT
ROOT.EnableImplicitMT() # enable multi-threading

Afterwards, we set a seed for PYTHIA8 and the number of events we want to generate.

In [ ]:
seed = 22
nEvents = 2000000

The next cell contains configuration for data generation on the Sunrise cluster at FNSPE CTU in Prague (which uses the PBS batch system) via splitting the generation into multiple jobs, e.g., generate 100 000 000 events through 100 jobs with each one generating 1 000 000 events. The job number setting (i.e., PBS_ARRAY_INDEX) should be modified to match the correct variable when used on a different cluster. It is assumed that the jobs are submitted via a batch job array on Sunrise: `qsub -q short -J 0-99 generate_pp510.sh`. This command submits 100 jobs into the short queue (maximum walltime of 2 hours for each job). Note that this notebook needs to be first converted into a .py script via the `jupyter nbconvert --to script pythia8_generate-tree.ipynb` command, and the `generate_pp510.sh` script checked and modified to match your work environment.

Nevertheless, this notebook **can be used locally**. In such case, the job number is always set to zero and the next cell can be ignored.

In [ ]:
# extract the job ID
try:
    jobID = int(os.environ.get('PBS_ARRAY_INDEX')) # CHANGE THIS to match the variables of the used batch system (PBS is used on Sunrise)
except:
    print("Job ID extraction failed! Assuming local generation from now on.")
    jobID = 0

# define the PYTHIA8 seed for the current job
seedJob = seed + jobID

# set the initial event ID: this ensures that the detector-effect simulator smears every events differently for each job because the smearing seed is directly set by event ID
# for example: if we generate 1 000 000 events in each job, the first job with jobID = 0 will have events labeled as 0, 1, 2, ..., 999 999; the second job with jobID=1 will have events labeled as 1 000 000, 1 000 001, 1 000 002, ...
nEventInit = jobID * nEvents

Then, we call the C++ generateEvents(nEvents, seed, outFileName) function with PYTHIA8 MC generator configured for $\sqrt{s} = 510 \: \mathrm{GeV}$ pp collisions with the Detroit tune for RHIC. The function stores the generated events into a TTree and saves everything into a .root file.

In [ ]:
# define the directory and file path
outDir = f"data/{seed}"
os.makedirs(outDir, exist_ok=True) # creates the directory if it doesn't exist, otherwise does nothing
outFileName = f"{outDir}/events{seedJob}.root"

# load the generateEvents(nEvents, seed, outFileName) function via the ROOT's gInterpreter
ROOT.gInterpreter.ProcessLine('#include "cpp/generate_events.cpp"')

# generate events
ROOT.generateEvents(nEvents, nEventInit, seed, outFileName)

### Simulating Detector Effects

Now, we have the simulated particle collisions from PYTHIA8 (the "true" data) and would like to obtain a data set similar to the ones we measure at STAR. In order to do so, we must simulate the detector effects of the STAR detector.

We are using RDataFrame for work with the data. We define a new data frame containing 4-vectors (tracks) of $p_\mathrm{T}$, $\eta$, $\phi$, mass using LorentzVector (PtEtaPhiMVector).

In [ ]:
# connect RDataFrame to the TTree in events{seed}.root
df = ROOT.RDataFrame("events", f"data/{seed}/events{seedJob}.root") 

# create tracks -- put pT, eta, phi, mass into a LorentzVector
dfTracks = df.Define("tracks", "ROOT::VecOps::Construct<ROOT::Math::PtEtaPhiMVector> (pT, eta, phi, mass)") # create tracks -- put pT, eta, phi, mass into a LorentzVector

!!! SO FAR, THE RESOLUTION AND EFFICIENCY FUNCTIONS ARE BASED FOR PIONS -- I NEED TO ADD OTHER PARTICLES IN THE FUTURE !!!

Firstly, we simulate the **efficiency of TPC and TOF**.

In [ ]:
# load the function TPCacceptance(Nch, pT) which randomly selects particles in an event based on an experimentally determined TPC's efficiency function (!!! now it's constant for all pT values, but will be modified to match experiments in the future) and a function TPCandTOFacceptance(Nch, pT, TPCaccepted) which works similarly, and also simulates efficiency of TOF
ROOT.gInterpreter.ProcessLine('#include "cpp/TPCandTOFefficiency.cpp"')

dfAccepted = (
    dfTracks.Define("TPCaccepted", "TPCmask(tracks.size(), pT, eventID)") # define a new column with events accepted (1) and rejected (0) by TPC
            .Define("TPCandTOFaccepted", "TPCandTOFmask(tracks.size(), pT, eventID, TPCaccepted)") # define a new column with events accepted (1) and rejected (0) by both TPC and TOF
            .Define("acceptedTracks", "tracks[TPCandTOFaccepted]") # define a new column to store only accepted tracks
)

Next, we need to smear $p_\mathrm{T}$, $\phi$, and $\eta$ to simulate **imperfect resolution of the detector**.

In [ ]:
ROOT.gInterpreter.ProcessLine('#include "cpp/smearing.cpp"') # load C++ function which smears the accepted tracks

dfSmearedRaw = (
    dfAccepted.Define("smearedTracksRaw", "smearing(acceptedTracks, eventID)")
)

We also apply the midrapidity cut $|\eta| < 1$ and the $p_\mathrm{T} > 0.2 \: \mathrm{GeV}$ cut.

In [ ]:
ROOT.gInterpreter.ProcessLine('#include "cpp/extract_components.cpp"') # load C++ functions which extract components of our tracks

dfSmeared = (
    dfSmearedRaw.Define("pTSmearedRaw", "getpT(smearedTracksRaw)") # extract pT
                .Define("etaSmearedRaw", "getEta(smearedTracksRaw)") # extract eta
                .Define("smearedAccepted", "pTSmearedRaw > 0.2 && abs(etaSmearedRaw) < 1.0") # define a new column containing masks for each event, which assign 1 to particles which passed both the pT and midrapidity cut, 0 otherwise
                .Define("smearedTracks", "smearedTracksRaw[smearedAccepted]") # redefine the smearedTracks column to keep only particles which passed both cuts
                .Define("pTSmeared", "pTSmearedRaw[smearedAccepted]")
                .Define("etaSmeared", "etaSmearedRaw[smearedAccepted]")
                .Define("phiSmeared", "getPhi(smearedTracks)")
)

In the end, we extract each 4-vector component into columns and save these columns into a new tree composed of smeared data.

In [ ]:
# save the smeared tree into a .root file
opts = ROOT.RDF.RSnapshotOptions() # ensure that events_smeared.root file is recreated every time this notebook is ran
opts.fMode = "RECREATE"
dfSmeared.Snapshot("events_smeared", f"data/{seed}/events{seedJob}_smeared.root", ['eventID', 'pT', 'eta', 'phi', 'mass', 'TPCaccepted', 'TPCandTOFaccepted', 'pTSmeared', 'etaSmeared', 'phiSmeared'], opts)